In [20]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from scipy import stats
stats.chisqprob = lambda chisq, df: stats.chi2.sf(chisq, df)

In [21]:
test = pd.read_csv(r"C:\Users\user\OneDrive\Dokumente\DATA SCIENCE\2.03 Model Testing dataset.csv")

In [22]:
test.head()

,SAT,Admitted,Gender
0,1323,No,Male
1,1725,Yes,Female
2,1762,Yes,Female
3,1777,Yes,Male
4,1665,No,Male


In [23]:
# Binary encoding (same mapping as training)
test["Admitted"] = test["Admitted"].map({"Yes": 1, "No": 0})
test["Gender"] = test["Gender"].map({"Male": 1, "Female": 0})

In [35]:
# here we are defining our target and independent variables 
y = test['Admitted']
x1 = test[['SAT','Gender']]

In [36]:
x = sm.add_constant(x1)
reg_log = sm.Logit(y,x)
results_log = reg_log.fit()
# Get the regression summary
results_log.summary()

Optimization terminated successfully.
         Current function value: 0.126762
         Iterations 11


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:               Admitted   No. Observations:                   19
Model:                          Logit   Df Residuals:                       16
Method:                           MLE   Df Model:                            2
Date:                Tue, 03 Feb 2026   Pseudo R-squ.:                  0.7967
Time:                        15:47:06   Log-Likelihood:                -2.4085
converged:                       True   LL-Null:                       -11.849
Covariance Type:            nonrobust   LLR p-value:                 7.940e-05
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -96.0385     75.475     -1.272      0.203    -243.966      51.889
SAT            0.0629      0.049      1.273      0.203      -0.034       0.160
Gender       -11.9067      9.605     -1.240      0.215     -30.732       6.918
==============================================================================

Possibly complete quasi-separation: A fraction 0.68 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

In [44]:
test_actual = test['Admitted']
test_data = test.drop(['Admitted'],axis=1)
test_data = sm.add_constant(test_data)
#test_data = test_data[x.columns.values]
test_data

,const,SAT,Gender
0,1.0,1323,1
1,1.0,1725,0
2,1.0,1762,0
3,1.0,1777,1
4,1.0,1665,1
5,1.0,1556,0
6,1.0,1731,0
7,1.0,1809,0
8,1.0,1930,0
9,1.0,1708,1


In [45]:
def confusion_matrix(data,actual_values,model):
    
        pred_values = model.predict(data)
        bins=np.array([0,0.5,1])
        cm = np.histogram2d(actual_values, pred_values, bins=bins)[0]
        accuracy = (cm[0,0]+cm[1,1])/cm.sum()
        return cm, accuracy

In [46]:
cm = confusion_matrix(test_data,test_actual,results_log)
cm

(array([[ 5.,  1.],
        [ 1., 12.]]),
 np.float64(0.8947368421052632))

In [47]:
cm_df = pd.DataFrame(cm[0])
cm_df.columns = ['Predicted 0','Predicted 1']
cm_df = cm_df.rename(index={0: 'Actual 0',1:'Actual 1'})
cm_df

,Predicted 0,Predicted 1
Actual 0,5.0,1.0
Actual 1,1.0,12.0
